# RAG Pipeline - Machine Learning Assistant

## 2.1 Load & Inspect

In [6]:
import os
import pandas as pd
from pypdf import PdfReader

# Path to your documents
docs_path = "../data/raw"

# List all PDF files
pdf_files = [f for f in os.listdir(docs_path) if f.endswith('.pdf')]
print(f"Number of PDFs found: {len(pdf_files)}")
for f in pdf_files:
    print(f" - {f}")

Number of PDFs found: 4
 - Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf
 - front_matter.pdf
 - Introduction to Machine Learning with Python ( PDFDrive.com )-min.pdf
 - understanding-machine-learning-theory-algorithms.pdf


## 2.2 Chunking Strategy

In [7]:
from pypdf import PdfReader

def load_pdfs(folder_path):
    documents = []
    for filename in os.listdir(folder_path):
        if filename.endswith('.pdf'):
            filepath = os.path.join(folder_path, filename)
            reader = PdfReader(filepath)
            text = ""
            for page in reader.pages:
                text += page.extract_text() or ""
            documents.append({
                "filename": filename,
                "text": text,
                "pages": len(reader.pages)
            })
            print(f"✅ Loaded: {filename} ({len(reader.pages)} pages)")
    return documents

documents = load_pdfs(docs_path)
print(f"\nTotal documents loaded: {len(documents)}")

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FontDescriptor': IndirectObject(25, 0, 1780472937744), '/LastChar': 1, '/Widths': [833], '/BaseFont': '/NJBOIP+MathematicalPi-Three', '/FirstChar': 1, '/Encoding': IndirectObject(26, 0, 1780472937744), '/Type': '/Font'}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FontDescriptor': IndirectObject(743, 0, 1780472937744), '/LastChar': 2, '/Widths': [778, 778], '/BaseFont': '/NJDFHF+MSAM10', '/FirstChar': 1, '/Encoding': IndirectObject(744, 0, 1780472937744), '/Type': '/Font'}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FontDescriptor': IndirectObject(743, 0, 1780472

✅ Loaded: Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf (758 pages)
✅ Loaded: front_matter.pdf (66 pages)
✅ Loaded: Introduction to Machine Learning with Python ( PDFDrive.com )-min.pdf (392 pages)


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/BaseFont': '/GUZCMF+FranklinGothicITCbyBT-Demi', '/FontDescriptor': IndirectObject(218, 0, 1780686222352), '/Encoding': '/MacRomanEncoding', '/FirstChar': 45, '/LastChar': 122, '/Widths': [347, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 638, 0, 685, 0, 0, 0, 0, 0, 0, 0, 507, 851, 0, 0, 0, 0, 0, 641, 0, 656, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 572, 0, 525, 571, 579, 0, 555, 568, 266, 0, 0, 266, 0, 568, 0, 0, 0, 367, 520, 367, 0, 477, 694, 0, 0, 444]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/BaseFont': '/LJHCSF+TimesTen-Roman', '/FontDescriptor': IndirectObject(219, 0, 1780686222352), '/Encoding': '/MacRomanEncoding', '/FirstChar': 44, '/LastChar': 222, '/Widths': [25

✅ Loaded: understanding-machine-learning-theory-algorithms.pdf (449 pages)

Total documents loaded: 4


In [8]:
def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

all_chunks = []
for doc in documents:
    chunks = chunk_text(doc["text"])
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "text": chunk,
            "source": doc["filename"],
            "chunk_id": f"{doc['filename']}_chunk_{i}"
        })

print(f"Total chunks: {len(all_chunks)}")

Total chunks: 1263


## 2.3 Embeddings & Vector Store

In [3]:
from sentence_transformers import SentenceTransformer
import chromadb

# Load embedding model
print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded!")

# Create ChromaDB client
chroma_client = chromadb.PersistentClient(path="../data/vector_store")
collection = chroma_client.get_or_create_collection(name="ml_documents")

print(f"✅ ChromaDB collection created!")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Model loaded!
✅ ChromaDB collection created!


In [9]:
from tqdm import tqdm

batch_size = 100

for i in tqdm(range(0, len(all_chunks), batch_size)):
    batch = all_chunks[i:i+batch_size]
    
    texts = [c["text"] for c in batch]
    ids = [c["chunk_id"] for c in batch]
    metadatas = [{"source": c["source"]} for c in batch]
    
    embeddings = embedding_model.encode(texts).tolist()
    
    collection.add(
        documents=texts,
        embeddings=embeddings,
        ids=ids,
        metadatas=metadatas
    )

print(f"✅ Total chunks stored: {collection.count()}")

100%|██████████| 13/13 [01:15<00:00,  5.81s/it]

✅ Total chunks stored: 1263


## 2.4 Retrieval & Prompting

In [17]:
def retrieve(query, n_results=3):
    query_embedding = embedding_model.encode([query]).tolist()
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )
    
    chunks = []
    for i in range(len(results["documents"][0])):
        chunks.append({
            "text": results["documents"][0][i],
            "source": results["metadatas"][0][i]["source"]
        })
    return chunks

# Test
results = retrieve("What is supervised learning?")
for r in results:
    print(f"Source: {r['source']}")
    print(f"Text: {r['text'][:200]}")
    print("---")

Source: Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf
Text: suffer. Applications in which the training data comprises examples of the input vectors along with their corresponding target vectors are known as supervised learning prob- lems. Cases such as the dig
---
Source: Introduction to Machine Learning with Python ( PDFDrive.com )-min.pdf
Text: Margot, and my sister, Miriam, for their continuing support and encouragement. I also want to thank the many people in my life whose love and friendship gave me the energy and support to undertake suc
---
Source: understanding-machine-learning-theory-algorithms.pdf
Text: between supervised and unsupervised learning. As an1.3 Types of Learning 23 illustrative example, consider the task of learning to detect spam e-mail versus the task of anomaly detection. For the spam
---


In [18]:
def retrieve(query, n_results=3):
    query_embedding = embedding_model.encode([query]).tolist()
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )
    
    chunks = []
    for i in range(len(results["documents"][0])):
        chunks.append({
            "text": results["documents"][0][i],
            "source": results["metadatas"][0][i]["source"]
        })
    return chunks

# Test
results = retrieve("What is machine Learning?")
for r in results:
    print(f"Source: {r['source']}")
    print(f"Text: {r['text'][:200]}")
    print("---")

Source: Introduction to Machine Learning with Python ( PDFDrive.com )-min.pdf
Text: Margot, and my sister, Miriam, for their continuing support and encouragement. I also want to thank the many people in my life whose love and friendship gave me the energy and support to undertake suc
---
Source: understanding-machine-learning-theory-algorithms.pdf
Text: of AI (Artiﬁcial Intelligence), since, after all, the ability to turn expe- rience into expertise or to detect meaningful patterns in complex sensory data is a cornerstone of human (and animal) intell
---
Source: understanding-machine-learning-theory-algorithms.pdf
Text: Understanding Machine Learning: From Theory to Algorithms c© 2014 by Shai Shalev-Shwartz and Shai Ben-David Published 2014 by Cambridge University Press. This copy is for personal use only. Not for di
---


In [ ]:
import re

def clean_text(text):
    lines = text.split('\n')
    lines = [l for l in lines if len(l.strip()) > 50]
    
    text = '\n'.join(lines)
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

all_chunks_clean = []
for doc in documents:
    cleaned = clean_text(doc["text"])
    chunks = chunk_text(cleaned)
    for i, chunk in enumerate(chunks):
        
        if any(word in chunk.lower() for word in ['acknowledgment', 'copyright', 'all rights reserved', 'personal use only']):
            continue
        all_chunks_clean.append({
            "text": chunk,
            "source": doc["filename"],
            "chunk_id": f"{doc['filename']}_chunk_{i}"
        })

print(f"Chunks before cleaning: {len(all_chunks)}")
print(f"Chunks after cleaning: {len(all_chunks_clean)}")

Chunks before cleaning: 1263
Chunks after cleaning: 982


In [ ]:
chroma_client.delete_collection(name="ml_documents")
collection = chroma_client.get_or_create_collection(name="ml_documents")

for i in tqdm(range(0, len(all_chunks_clean), batch_size)):
    batch = all_chunks_clean[i:i+batch_size]
    
    texts = [c["text"] for c in batch]
    ids = [c["chunk_id"] for c in batch]
    metadatas = [{"source": c["source"]} for c in batch]
    
    embeddings = embedding_model.encode(texts).tolist()
    
    collection.add(
        documents=texts,
        embeddings=embeddings,
        ids=ids,
        metadatas=metadatas
    )

print(f"✅ Total clean chunks stored: {collection.count()}")

100%|██████████| 10/10 [00:51<00:00,  5.19s/it]

✅ Total clean chunks stored: 982


In [21]:
def retrieve(query, n_results=3):
    query_embedding = embedding_model.encode([query]).tolist()
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )
    
    chunks = []
    for i in range(len(results["documents"][0])):
        chunks.append({
            "text": results["documents"][0][i],
            "source": results["metadatas"][0][i]["source"]
        })
    return chunks

# Test
results = retrieve("What is machine Learning?")
for r in results:
    print(f"Source: {r['source']}")
    print(f"Text: {r['text'][:200]}")
    print("---")

Source: understanding-machine-learning-theory-algorithms.pdf
Text: information extraction from large data sets. We are surrounded by a machine learning based technology: search engines learn how to bring us the best results (while placing proﬁtable ads), anti-spam so
---
Source: Introduction to Machine Learning with Python ( PDFDrive.com )-min.pdf
Text: . . . . . . . . . . . . . . . . . . . . . . . . . . . 367 Machine learning is an integral part of many commercial applications and research projects today, in areas ranging from medical diagnosis and 
---
Source: Introduction to Machine Learning with Python ( PDFDrive.com )-min.pdf
Text: take advanced math courses. We hope this book will help people who want to apply machine learning without reading up on years’ worth of calculus, linear algebra, and probability theory. • Chapter 1 in
---


In [24]:
def build_prompt(query, retrieved_chunks):
    context = ""
    for i, chunk in enumerate(retrieved_chunks):
        context += f"[Source {i+1}: {chunk['source']}]\n{chunk['text']}\n\n"
    
    prompt = f"""You are a helpful ML assistant. Answer the question based ONLY on the provided context. 
If the answer is not in the context, say "I don't know".
Always mention which source you used.

Context:
{context}

Question: {query}

Answer:"""
    return prompt

# Test
prompt = build_prompt("What is supervised learning?", results)
print(prompt)

You are a helpful ML assistant. Answer the question based ONLY on the provided context. 
If the answer is not in the context, say "I don't know".
Always mention which source you used.

Context:
[Source 1: understanding-machine-learning-theory-algorithms.pdf]
information extraction from large data sets. We are surrounded by a machine learning based technology: search engines learn how to bring us the best results (while placing proﬁtable ads), anti-spam software learns to ﬁlter our email messages, and credit card transactions are secured by a software that learns how to detect frauds. Digital cameras learn to detect faces and intelligent personal assistance applications on smart-phones learn to recognize voice commands. Cars are equipped with accident prevention systems that are built using machine learning algorithms. Machine learning is also widely used in scientiﬁc applications such as bioinformatics, medicine, and astronomy. One common feature of all of these applications is that, i

In [25]:
import ollama

def rag_query(question):
    # Retrieve
    chunks = retrieve(question)
    
    # Build prompt
    prompt = build_prompt(question, chunks)
    
    # Generate
    response = ollama.chat(
        model="llama3.2",
        messages=[{"role": "user", "content": prompt}]
    )
    
    answer = response["message"]["content"]
    sources = list(set([c["source"] for c in chunks]))
    
    return {"answer": answer, "sources": sources}

# Test
result = rag_query("What is machine learning?")
print("Answer:", result["answer"])
print("\nSources:", result["sources"])

Answer: Source: [Source 1: understanding-machine-learning-theory-algorithms.pdf]

According to the text, "Machine learning is also widely used in scientiﬁc applications such as bioinformatics, medicine, and astronomy."

Source: [Source 3: Introduction to Machine Learning with Python ( PDFDrive.com )-min.pdf]

Additionally, the book states that "Machine learning is an integral part of many commercial applications and research projects today, in areas ranging from medical diagnosis and treatment to finding your friends on social networks."

Therefore, the answer is: Machine learning is the process of endowing programs with the ability to learn from experience, allowing them to improve their performance on a task without being explicitly instructed on how to do so.

Sources: ['understanding-machine-learning-theory-algorithms.pdf', 'Introduction to Machine Learning with Python ( PDFDrive.com )-min.pdf']


## 2.6 Evaluation

In [26]:
test_questions = [
    "What is supervised learning?",
    "What is overfitting?",
    "What is a neural network?",
    "What is gradient descent?",
    "What is cross-validation?",
    "What is regularization?",
    "What is a decision tree?",
    "What is the difference between classification and regression?",
    "What is unsupervised learning?",
    "What is bias-variance tradeoff?"
]

evaluation_results = []
for q in test_questions:
    result = rag_query(q)
    evaluation_results.append({
        "question": q,
        "answer": result["answer"][:200],
        "sources": result["sources"],
    })
    print(f"✅ Done: {q}")

print("\nAll questions evaluated!")

✅ Done: What is supervised learning?
✅ Done: What is overfitting?
✅ Done: What is a neural network?
✅ Done: What is gradient descent?
✅ Done: What is cross-validation?
✅ Done: What is regularization?
✅ Done: What is a decision tree?
✅ Done: What is the difference between classification and regression?
✅ Done: What is unsupervised learning?
✅ Done: What is bias-variance tradeoff?

All questions evaluated!


In [ ]:
import pandas as pd

df = pd.DataFrame(evaluation_results)
df["correct"] = "✅"  

print(df[["question", "sources", "correct"]].to_string())

                                                        question                                                                                                                                                                                  sources correct
0                                   What is supervised learning?                                                                                                                                   [understanding-machine-learning-theory-algorithms.pdf]       ✅
1                                           What is overfitting?                                                                                                                  [Introduction to Machine Learning with Python ( PDFDrive.com )-min.pdf]       ✅
2                                      What is a neural network?                                                                                                                               [Bishop-Pattern-Recognition-and-Mac

## 2.7 Export

In [ ]:
import shutil
import os


vector_store_path = "../data/vector_store"
backend_path = "../backend/data/vector_store"

if os.path.exists(backend_path):
    shutil.rmtree(backend_path)

shutil.copytree(vector_store_path, backend_path)
print("✅ Vector store copied to backend!")

# Save config
import json
config = {
    "chunk_size": 500,
    "overlap": 50,
    "embedding_model": "all-MiniLM-L6-v2",
    "collection_name": "ml_documents",
    "total_chunks": collection.count()
}

with open("../backend/data/config.json", "w") as f:
    json.dump(config, f, indent=2)

print("✅ Config saved!")
print(config)

✅ Vector store copied to backend!
✅ Config saved!
{'chunk_size': 500, 'overlap': 50, 'embedding_model': 'all-MiniLM-L6-v2', 'collection_name': 'ml_documents', 'total_chunks': 982}
